In [2]:

import d3rlpy
from d3rlpy.dataset import InfiniteBuffer, ReplayBuffer
from d3rlpy.algos.transformer.decision_transformer import DTConstantRTGforFQE,DecisionTransformer
from d3rlpy.ope.fqe import FQE,FQEConfig
import time

on_server=True

prefix = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/" if on_server else "/home/julian/programming/cloned_repos/repos_for_master_thesis/"
dt_model = prefix + "d3rlpy/experiments/exp01_original_dt/slurm_files/d3rlpy_logs/gpu_array/DT_hopper-medium-expert-v2_1_20250710202727/model_epoch_12.d3"

device = "cuda:0" if on_server else "cpu"
device = "cpu"
dt_algo = d3rlpy.load_learnable(dt_model,device=device)

In [6]:
from types import SimpleNamespace
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

args = SimpleNamespace(
    dataset="hopper-medium-expert-v2",
    context_size=20,
    #model_file="/gpfs/data/fs72297/jklotz/programming/cloned_repos/forked_repos_for_master_thesis/d3rlpy/experiments/exp02_qdt/d3rlpy_logs/CQL_Hopper-v4_1_20250709212343/model_4950.d3",
    model_file=None,
    q_learning_type="cql",
    seed=1,
    num_action_samples=10,
    gpu=0,
    compile = False
)
# parser = argparse.ArgumentParser()
# parser.add_argument("--dataset", type=str, default="hopper-medium-v2")
# parser.add_argument("--context_size", type=int, default=20)
# parser.add_argument("--model_file", type=str, default=None)
# parser.add_argument(
#     "--q_learning_type",
#     type=str,
#     default="cql",
#     choices=["cql", "iql"],
# )
# parser.add_argument("--seed", type=int, default=1)
# parser.add_argument("--num_action_samples", type=int, default=10)
# parser.add_argument("--gpu", type=int)
# args = parser.parse_args()

if "halfcheetah" in args.dataset:
    env_name = "HalfCheetah-v4"
    target_return = 6000
elif "hopper" in args.dataset:
    env_name = "Hopper-v4"
    target_return = 3600
elif "walker" in args.dataset:
    env_name = "Walker2d-v4"
    target_return = 5000
    
pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{args.dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")

from d3rlpy.dataset import InfiniteBuffer, ReplayBuffer, FIFOBuffer


buffer_impl = FIFOBuffer(100000)
dataset = ReplayBuffer(buffer=buffer_impl, episodes=episodes)

Loaded 3213 episodes.
2025-08-08 10:57.05 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-08-08 10:57.05 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-08-08 10:57.05 [info     ] Action size has been automatically determined. action_size=3


In [18]:
from d3rlpy.algos import DecisionTransformerConfig
from d3rlpy.algos.transformer import StatefulTransformerWrapper
import numpy as np

def evaluate_dt_on_dataset(algo, dataset, target_return):
    """
    algo: trained DT algorithm (already fitted).
    dataset: list of episodes (Episode objects from d3rlpy dataset).
    target_return: float, desired return-to-go.
    """
    #wrapper = StatefulTransformerWrapper(algo, target_return=target_return)
    wrapper = algo.as_stateful_wrapper(target_return=target_return,action_sampler=None)
    all_preds = []
    all_labels = []

    for i,episode in enumerate(dataset.episodes):
        if i>10:
            break
        wrapper.reset()
        obs_seq = episode.observations
        act_seq = episode.actions
        rew_seq = episode.rewards

        reward = 0.0
        for t in range(len(obs_seq)):
            # DT predicts next action given history up to now
            pred_action = wrapper.predict(obs_seq[t], reward)

            # store prediction and true action
            all_preds.append(pred_action)
            all_labels.append(act_seq[t])

            # next reward is from dataset
            reward = float(rew_seq[t])

    # Example metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # For discrete actions:
    acc = np.mean(all_preds == all_labels)

    # For continuous actions, use MSE:
    mse = np.mean((all_preds - all_labels) ** 2)

    return {"accuracy": acc, "mse": mse}


In [19]:
evaluate_dt_on_dataset(dt_algo,dataset,target_return)

/tmp/ipykernel_408761/2152298717.py:34: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  reward = float(rew_seq[t])


{'accuracy': 3.112646683474959e-05, 'mse': 0.07332972}